# 下载检查：voraus-AD 100 Hz

本 Notebook 是 Codex 执行的只读检查记录，供复查使用，不代表用户已独立实现。原始数据不改写。

核对全部元数据与计划使用的 18 路信号；其他信号仅检查字段结构。官方加载器以 setting=72 (PRE_A) 选训练数据，其余为测试数据。正常类别编码为 12。

临时 pyarrow 组件位于工作区 tmp/robot-data-audit-deps；使用 Python 3.12 内核可复跑。没有训练模型。


## 1. 文件结构和指纹


In [1]:
from pathlib import Path
import sys, hashlib, json, platform
from datetime import datetime, timezone
sys.path.insert(0, r"D:\10_AI_Industrial_Engineer\tmp\robot-data-audit-deps")
import pyarrow
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
path = Path(r"D:\10_AI_Industrial_Engineer\Project_01_ML\data\raw\voraus-ad-dataset-100hz.parquet")
pf = pq.ParquetFile(path)
with path.open("rb") as stream:
    fingerprint = hashlib.file_digest(stream, "sha256").hexdigest()
result = {"checked_at_utc": datetime.now(timezone.utc).isoformat(),
          "file": str(path), "bytes": path.stat().st_size, "sha256": fingerprint,
          "rows": pf.metadata.num_rows, "columns": pf.metadata.num_columns,
          "schema": {f.name: str(f.type) for f in pf.schema_arrow},
          "versions": {"python": platform.python_version(), "pyarrow": pyarrow.__version__,
                       "pandas": pd.__version__, "numpy": np.__version__}}
print("File bytes:", result["bytes"], "Rows:", result["rows"], "Columns:", result["columns"])
print("SHA-256:", fingerprint)


File bytes: 1115942833 Rows: 2321690 Columns: 137
SHA-256: c90ab1c78af52651b954d41787f7e89d750f0a128b57600b0e5ceec22621f704


## 2. 按完整作业统计及核对官方划分


In [2]:
meta_columns = ["sample", "time", "anomaly", "category", "setting", "action", "active"]
meta = pf.read(columns=meta_columns).to_pandas()
groups = meta.groupby("sample", sort=True)
cycles = groups[["anomaly", "category", "setting"]].first()
cycles["points"] = groups.size()
cycles["time_first_s"] = groups["time"].first()
cycles["time_last_s"] = groups["time"].last()
cycles["official_set"] = np.where(cycles["setting"] == 72, "train", "test")
train = cycles["official_set"].eq("train")
result["counts"] = {
    "cycles": len(cycles), "normal": int((~cycles["anomaly"]).sum()),
    "anomaly": int(cycles["anomaly"].sum()), "official_train_normal": int((train & ~cycles["anomaly"]).sum()),
    "official_train_anomaly": int((train & cycles["anomaly"]).sum()),
    "official_test_normal": int((~train & ~cycles["anomaly"]).sum()),
    "official_test_anomaly": int((~train & cycles["anomaly"]).sum())}
result["category_cycle_counts"] = {str(k): int(v) for k, v in cycles["category"].value_counts().sort_index().items()}
result["counts_match_project_brief"] = result["counts"] == {"cycles": 2122, "normal": 1367,
    "anomaly": 755, "official_train_normal": 948, "official_train_anomaly": 0,
    "official_test_normal": 419, "official_test_anomaly": 755}
print(json.dumps(result["counts"], indent=2))


{
  "cycles": 2122,
  "normal": 1367,
  "anomaly": 755,
  "official_train_normal": 948,
  "official_train_anomaly": 0,
  "official_test_normal": 419,
  "official_test_anomaly": 755
}


## 3. 标签、时间与重复键检查


In [3]:
dt = groups["time"].diff().dropna()
label_columns = ["anomaly", "category", "setting"]
result["metadata_nulls"] = {k: int(v) for k, v in meta.isna().sum().items()}
result["inconsistent_cycles"] = {k: int(v) for k, v in (groups[label_columns].nunique(dropna=False) > 1).sum().items()}
result["duplicate_sample_time_rows"] = int(meta.duplicated(["sample", "time"]).sum())
block_ids = meta.loc[meta["sample"].ne(meta["sample"].shift()), "sample"]
result["sample_ids_in_multiple_blocks"] = int(block_ids.duplicated().sum())
result["category_label_mismatch_cycles"] = int((cycles["anomaly"] != cycles["category"].ne(12)).sum())
result["time"] = {"finite": bool(np.isfinite(meta["time"]).all()),
    "nonpositive_intervals": int((dt <= 0).sum()), "min_interval_s": float(dt.min()),
    "median_interval_s": float(dt.median()), "max_interval_s": float(dt.max()),
    "intervals_outside_9_to_11_ms": int(((dt < 0.009) | (dt > 0.011)).sum()),
    "total_within_cycle_intervals": len(dt)}
result["points_per_cycle"] = {"min": int(cycles["points"].min()),
    "median": float(cycles["points"].median()), "max": int(cycles["points"].max())}
print("Metadata nulls:", result["metadata_nulls"])
print("Inconsistent cycles:", result["inconsistent_cycles"])
print("Duplicate sample/time rows:", result["duplicate_sample_time_rows"])
print("Time:", result["time"])
print("Points per cycle:", result["points_per_cycle"])


Metadata nulls: {'sample': 0, 'time': 0, 'anomaly': 0, 'category': 0, 'setting': 0, 'action': 0, 'active': 0}
Inconsistent cycles: {'anomaly': 0, 'category': 0, 'setting': 0}
Duplicate sample/time rows: 0
Time: {'finite': True, 'nonpositive_intervals': 0, 'min_interval_s': 0.00845193862915039, 'median_interval_s': 0.010066986083984375, 'max_interval_s': 0.01211094856262207, 'intervals_outside_9_to_11_ms': 3, 'total_within_cycle_intervals': 2319568}
Points per cycle: {'min': 986, 'median': 1096.0, 'max': 1164}


## 4. 逐批检查 18 路信号


In [4]:
signals = [f"{prefix}_{i}" for prefix in ["motor_iq", "joint_velocity", "torque_sensor_a"] for i in range(1, 7)]
result["missing_required_signals"] = sorted(set(signals) - set(pf.schema_arrow.names))
stats = {name: {"nulls": 0, "nonfinite": 0, "min": float("inf"), "max": float("-inf")} for name in signals}
for batch in pf.iter_batches(batch_size=65536, columns=signals):
    for name in signals:
        column = batch.column(name)
        values = column.to_numpy(zero_copy_only=False)
        finite = np.isfinite(values)
        stat = stats[name]
        stat["nulls"] += column.null_count
        stat["nonfinite"] += int((~finite).sum())
        if finite.any():
            stat["min"] = min(stat["min"], float(values[finite].min()))
            stat["max"] = max(stat["max"], float(values[finite].max()))
result["selected_signal_checks"] = stats
result["globally_constant_selected_signals"] = [name for name, stat in stats.items() if stat["min"] == stat["max"]]
print("Required signals:", len(signals), "Missing:", result["missing_required_signals"])
print("Nulls:", sum(s["nulls"] for s in stats.values()), "Nonfinite:", sum(s["nonfinite"] for s in stats.values()))
print("Globally constant selected signals:", result["globally_constant_selected_signals"])


Required signals: 18 Missing: []
Nulls: 0 Nonfinite: 0
Globally constant selected signals: []


## 5. 正常样例及保存检查结果


In [5]:
example_id = int(cycles.index[train & ~cycles["anomaly"]][0])
example = meta.loc[meta["sample"].eq(example_id)]
result["normal_example"] = {"sample": example_id, "points": len(example),
    "first_time_s": float(example["time"].iloc[0]), "last_time_s": float(example["time"].iloc[-1])}
result["first_file_cycle"] = {"sample": int(meta["sample"].iloc[0]), "anomaly": bool(meta["anomaly"].iloc[0])}
result["scope"] = "All rows of seven metadata fields and 18 selected signals were read; other signal columns were not fully audited. No model was trained and no split was newly created."
result["limitations"] = ["Local SHA-256 is a fingerprint; it was not compared with a publisher checksum.",
    "The 9-11 ms band is an inspection aid, not a validated rejection threshold.",
    "Signal units were not independently measured. Cross-cycle duplicate signal traces were not checked."]
result["official_sources"] = ["https://github.com/vorausrobotik/voraus-ad-dataset",
    "https://github.com/vorausrobotik/voraus-ad-dataset/blob/main/voraus_ad.py"]
report_path = path.parents[2] / "reports" / "download_check_2026-10-10.json"
report_path.write_text(json.dumps(result, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
print("Normal training example:", result["normal_example"])
print("First cycle in file:", result["first_file_cycle"])
print("Saved:", report_path)


Normal training example: {'sample': 755, 'points': 1069, 'first_time_s': 0.0, 'last_time_s': 10.753396987915039}
First cycle in file: {'sample': 0, 'anomaly': True}
Saved: D:\10_AI_Industrial_Engineer\Project_01_ML\reports\download_check_2026-10-10.json
